# comms.io daily Colab run

The SQLite working database is stored as a compressed, chunked snapshot in the
`data` branch under `state/`. This notebook downloads it with
`comms state unpack`, crawls, exports Parquet, and publishes both the website
data and the updated state snapshot back to `data`.

Nothing is stored in Google Drive. The notebook can run in a fresh Colab VM.

Add `GITHUB_TOKEN` in the Colab secrets panel. A fine-grained token with
`contents: write` is enough. The website reads the data branch through
raw.githubusercontent.com, so data updates do not require a Pages deploy.

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/Elfiawesome/comms.io.git"
REPO_DIR = Path("/content/comms.io")

if REPO_DIR.exists():
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

In [ ]:
import sys

venv = Path(".venv")
if not (venv / "bin" / "python").exists():
    !{sys.executable} -m venv .venv

PY = str(venv / "bin" / "python")
os.environ["PATH"] = str((venv / "bin").resolve()) + os.pathsep + os.environ.get("PATH", "")
!{PY} -m pip install --upgrade pip
!{PY} -m pip install -e .

# Optional: only needed if run_gallery_dl is enabled.
# !{PY} -m pip install gallery-dl

In [ ]:
from google.colab import userdata

os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
os.environ["COMMS_DATABASE"] = str(REPO_DIR / "work" / "comms.sqlite3")
os.environ["COMMS_EXPORT_DIR"] = str(REPO_DIR / "dist" / "data" / "v1")
os.environ["COMMS_GALLERY_DL_DIR"] = str(REPO_DIR / "work" / "gallery-dl")
os.environ["COMMS_TRIGGER_DEPLOY"] = "false"

print("database:", os.environ["COMMS_DATABASE"])
print("export:", os.environ["COMMS_EXPORT_DIR"])

In [ ]:
# Restore the full archive state from the data branch. If no state exists yet,
# the archiver starts with an empty SQLite database.
!{PY} -m comms state unpack --repo Elfiawesome/comms.io --branch data

## Optional: refresh the X/Twitter seed catalog

This uses the old gallery-dl workflow. It is optional because it depends on
your X cookies and gallery-dl continuing to work. Put `cookies.txt` in the
repository root if required by gallery-dl. The crawl also imports any JSON
already present in `work/gallery-dl`.

In [ ]:
# Uncomment to run gallery-dl before the crawl.
# !{PY} -m comms crawl --run-gallery-dl --limit 0 --publish

In [ ]:
# Normal daily run: discover a little, refresh stale artists, export Parquet +
# the chunked SQLite state, then publish the data branch.
!{PY} -m comms crawl --limit 0 --publish

In [ ]:
!{PY} -m comms status
!{PY} -m comms validate
!{PY} -m comms state status

## Tuning

Edit `config/comms.toml` in the repo (or pass CLI options) to change:

- `run.max_requests` and `run.max_runtime_minutes`
- `run.max_new_artists` and `run.max_refresh_artists`
- `network.requests_per_second`
- `sources.skeb.discover_pages`

If repeated 429s occur, lower the rate and concurrency first. The parser and
rate limiter are deliberately conservative on shared Colab IPs.